<a href="https://colab.research.google.com/github/wdpressplus-bigdata/wdpressplus-bigdata/blob/main/notebooks/7-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# p.273 List 7.2
import pathlib
import requests

def download_file(filename):
    prefix = 'https://github.com/wdpressplus-bigdata/uscrn/raw/main'
    # prefix = 'https://www1.ncdc.noaa.gov/pub/data/uscrn/products/subhourly01'
    r = requests.get(f"{prefix}/2020/{filename}")
    r.raise_for_status()
    path = pathlib.Path('./raw')
    path.mkdir(parents=True, exist_ok=True)
    with open(path / filename, 'wb') as f:
        f.write(r.content)
    print(f"Saved {path / filename}")

FILES = [
    'CRNS0101-05-2020-AK_Aleknagik_1_NNE.txt',
    'CRNS0101-05-2020-AK_Bethel_87_WNW.txt',
]
for filename in FILES:
    download_file(filename)

Saved raw/CRNS0101-05-2020-AK_Aleknagik_1_NNE.txt
Saved raw/CRNS0101-05-2020-AK_Bethel_87_WNW.txt


In [ ]:
# p.274
!pip install pandas

: 

: 

In [ ]:
import glob
import pandas as pd

def read_tables():
    for path in glob.glob('./raw/*.txt'):
        yield pd.read_table(path, delimiter='\s+', header=None, dtype='str')

df = pd.concat(read_tables())
df.head(2)

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
0,26656,20200101,0005,20191231,1505,3,-164.08,61.35,-20.8,0.0,...,C,0,80,0,-99.000,-9999.0,1005,0,6.69,0
1,26656,20200101,0010,20191231,1510,3,-164.08,61.35,-20.8,0.0,...,C,0,79,0,-99.000,-9999.0,1005,0,7.64,0


: 

: 

In [ ]:
print(df[0])
print(df.shape)


0        26656
1        26656
2        26656
3        26656
4        26656
         ...  
96175    23583
96176    23583
96177    23583
96178    23583
96179    23583
Name: 0, Length: 192360, dtype: object
(192360, 23)


: 

: 

In [ ]:
# p.275
df1 = df[[0, 1, 2, 8]]
df1.columns = ['wbanno', 'utc_date', 'utc_time', 'temperature']
df1.head(2)

,wbanno,utc_date,utc_time,temperature
0,26656,20200101,0005,-20.8
1,26656,20200101,0010,-20.8


: 

: 

In [ ]:
df2 = df1.copy()
df2.index = pd.to_datetime(df2['utc_date'] + df2['utc_time'])
df2.drop(columns=['utc_date', 'utc_time'], inplace=True)
df2.head(2)

,wbanno,temperature
2020-01-01 00:05:00,26656,-20.8
2020-01-01 00:10:00,26656,-20.8


: 

: 

In [ ]:
# p.276
df2.describe().T

,count,unique,top,freq
wbanno,192360,2,26656,96180
temperature,192360,569,0.3,1182


: 

: 

In [ ]:
df2['temperature'] = df2['temperature'].astype('float')
df2.describe().T

,count,mean,std,min,25%,50%,75%,max
temperature,192360.0,-9.160542,325.703007,-9999.0,-5.0,3.7,10.2,24.8


: 

: 

In [15]:
# p.277
df3 = df2.copy()
df3.loc[df3['temperature'] == -9999.0, 'temperature'] = None
df3.describe().T

NameError: name 'df2' is not defined

In [ ]:
# p.278
%pip install pyspark>=3.3


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: /Users/fukunagaatsushi/src/github.com/wdpressplus-bigdata/wdpressplus-bigdata/bigdata-venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


: 

: 

In [1]:
from pyspark.sql.session import SparkSession

# セキュリティマネージャ関連の依存関係をJVM オプションで回避
spark = SparkSession.builder \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/27 22:50:02 WARN Utils: Your hostname, atsushinoMacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.3.154 instead (on interface en0)
26/05/27 22:50:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 22:50:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [13]:
# p.279
rdd = spark.sparkContext.textFile('./raw/*')
rdd.take(2)

ConnectionRefusedError: [Errno 61] Connection refused

In [3]:
rdd.count()

192360

In [2]:
from datetime import datetime, timezone
from pyspark.sql import Row

def parse_line(line):
    f = line.split()
    wbanno = f[0]
    dt = datetime.strptime(f[1] + f[2], '%Y%m%d%H%M')
    dt = dt.replace(tzinfo=timezone.utc)
    temperature = None if f[8] == '-9999.0' else float(f[8])
    return Row(timestamp=dt, wbanno=wbanno, temperature=temperature)

rows = rdd.map(parse_line)
rows.take(2)

NameError: name 'rdd' is not defined

In [6]:
# p.280
df = rdd.map(parse_line).toDF()
df

DataFrame[timestamp: timestamp, wbanno: string, temperature: double]

In [7]:
spark.conf.set("spark.sql.session.timeZone", 'UTC')

In [8]:
df.show(2)

+-------------------+------+-----------+
|          timestamp|wbanno|temperature|
+-------------------+------+-----------+
|2020-01-01 00:05:00| 23583|      -17.5|
|2020-01-01 00:10:00| 23583|      -17.5|
+-------------------+------+-----------+
only showing top 2 rows


Traceback (most recent call last):
  File "/Users/fukunagaatsushi/src/github.com/wdpressplus-bigdata/wdpressplus-bigdata/bigdata-venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/fukunagaatsushi/src/github.com/wdpressplus-bigdata/wdpressplus-bigdata/bigdata-venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [9]:
# p.281
df.describe().show()

+-------+------------------+------------------+
|summary|            wbanno|       temperature|
+-------+------------------+------------------+
|  count|            192360|            192156|
|   mean|           25119.5|1.4450451716313355|
| stddev|1536.5039938292543|11.572590880335754|
|    min|             23583|             -32.0|
|    max|             26656|              24.8|
+-------+------------------+------------------+



In [5]:
df.createOrReplaceTempView('uscrn')

In [ ]:
query = '''
SELECT
  wbanno,
  min_by(timestamp, temperature) timestamp_min,
  min(temperature) t_min,
  max_by(timestamp, temperature) timestamp_max,
  max(temperature) t_max
FROM
  uscrn
GROUP by
  1
'''
spark.sql(query).show()

+------+-------------------+-----+-------------------+-----+
|wbanno|      timestamp_min|t_min|      timestamp_max|t_max|
+------+-------------------+-----+-------------------+-----+
| 23583|2020-02-02 01:15:00|-32.0|2020-08-17 09:20:00| 24.8|
| 26656|2020-02-10 00:15:00|-30.8|2020-05-31 08:05:00| 23.3|
+------+-------------------+-----+-------------------+-----+



26/05/25 10:10:12 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 987012 ms exceeds timeout 120000 ms
26/05/25 10:10:12 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/25 10:10:20 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o

In [12]:
# p.283
df.write.save('./uscrn-parquet')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/Users/fukunagaatsushi/src/github.com/wdpressplus-bigdata/wdpressplus-bigdata/notebooks/uscrn-parquet already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

In [8]:
!ls ./uscrn-parquet

_SUCCESS
part-00000-147aaf54-5136-488d-b64e-8a865dbc1270-c000.snappy.parquet
part-00001-147aaf54-5136-488d-b64e-8a865dbc1270-c000.snappy.parquet


In [3]:
from pyspark.sql.session import SparkSession

# セキュリティマネージャ関連の依存関係をJVM オプションで回避
spark = SparkSession.builder \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .getOrCreate()

df = spark.read.load('./uscrn-parquet')
df.groupBy('wbanno').avg('temperature').show()

+------+------------------+
|wbanno|  avg(temperature)|
+------+------------------+
| 23583|2.4658855466799405|
| 26656|0.4228013165020489|
+------+------------------+



In [10]:
# p.284
df = spark.read.load('./uscrn-parquet')
df1 = df.where("timestamp >= '2020-01-01' AND timestamp < '2020-04-01'")
df1.count()

52414

In [ ]:
df1.coalesce(1).write.save('./export', format='csv', header=True)

In [6]:
!ls ./export

_SUCCESS
part-00000-a6bf622a-92de-47f1-80b4-948a926b24f2-c000.csv


In [ ]:
!cp ./export/*.csv ~/Home/Desktop/

In [13]:
# p.285 Column
!pip install pyarrow

In [8]:
from pyspark.sql.session import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", 'UTC')

df = spark.read.load('./uscrn-parquet')
df1 = df.groupBy('timestamp').avg().toPandas()
df1.sort_values(by='avg(temperature)', ascending=False).head(2)

,timestamp,avg(temperature)
43454,2020-07-17 00:50:00,22.9
7468,2020-08-17 02:55:00,22.7


In [16]:
import pandas as pd
df = pd.read_parquet('./uscrn-parquet', use_pandas_metadata=False)

df1 = df.groupby('timestamp').mean()
df1.sort_values(by='temperature', ascending=False).head(2)

ArrowKeyError: A type extension with name pandas.period already defined

26/05/28 00:02:33 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 915832 ms exceeds timeout 120000 ms
26/05/28 00:02:34 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/28 00:02:40 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o